# SPICE — Phase 1 Pre-train

> SPICE: Sequence-Protein Interaction under Conditional Environments
> This notebook runs the Pre-train with **TensorFlow**: dynamic Transformer + AdaLN + Head A (Cα coords) + a distogram head, supervised by binned distogram cross-entropy.

Run order: 1️⃣ install deps → 2️⃣ get code → 3️⃣ build TFRecord → 4️⃣ train → 5️⃣ visualize.

In [ ]:
# ① Install dependencies (China pip mirror) + check/enable GPU
# 版本钉死：与保存 checkpoint 的环境一致（TF 2.21 + Keras 3.15）。
# 不同 Keras 版本的优化器 checkpoint 布局/dtype 不同（step_counter int64→float32 等），
# 跨平台续训会 RestoreV2 报错；钉死版本后 optimizer 状态即可完整恢复。
# 注意：Kaggle 上清华镜像可能不通——跑不动就把 "-i https://pypi.tuna.tsinghua.edu.cn/simple" 去掉用默认 PyPI。
!pip install -q -i https://pypi.tuna.tsinghua.edu.cn/simple "tensorflow==2.21" "keras==3.15" datasets huggingface_hub pyarrow polars pyyaml tqdm

import tensorflow as tf
import keras

# HF download endpoint is chosen automatically (handled in dataset.py; do NOT set HF_ENDPOINT here):
#   - Colab: the VM runs on Google Cloud, so the official huggingface.co is reachable directly
#     (pointing at the China mirror would actually fail to download files)
#   - Local dev (in China): use data.hf_endpoint in configs/pretrain.yaml (hf-mirror.com)
from spice_pre.config import load_config
from spice_pre.keras_utils import setup_gpu

cfg = load_config("configs/pretrain.yaml")
setup_gpu(cfg.train.use_gpu, cfg.train.gpu_mem_growth, cfg.train.gpu_devices)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", "ON" if cfg.train.use_gpu else "OFF (forced CPU)")
print("Available devices:", gpus if gpus else ["CPU"])

ModuleNotFoundError: No module named 'spice_pre'

## 2. Build TFRecord

Downloads the `SPICE-Protein/spice_protein` parquet files from HF, cleans them (keeping only structures with environment labels and valid lengths), then writes TFRecords.

> For debugging use `max_shards=2` to run through quickly; for the real training set it to `0` (all shards, ~1.4 GB download).

In [ ]:
from spice_pre.config import load_config
from spice_pre.data.dataset import build_tfrecords

cfg = load_config("configs/pretrain.yaml")
cfg.data.max_shards = 0          # production: all shards (use_env_filtered=false uses the full ~45k; ~1.4 GB download)
cfg.data.use_env_filtered = False

n = build_tfrecords(cfg)
print("TFRecord record count:", n)

## 3. Train

Trains the backbone + Head A + distogram head. Main loss: **binned distogram cross-entropy** + a small pairwise-coordinate RMSE auxiliary term. Kabsch RMSD is reported as a validation metric only.

> Debug: `epochs=2, max_steps=300` finishes in a few minutes; production: `epochs=30, max_steps=0`.

In [ ]:
from spice_pre.train_pretrain import train

cfg.train.epochs = 30
cfg.train.max_steps = 0          # production: run all epochs

train(cfg)

## 4. Visualize

Loads the best weights and compares the predicted Cα backbone against the ground-truth Cα backbone on one validation sample.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from spice_pre.config import load_config
from spice_pre.models import SPICEPretrainModel
from spice_pre.data.dataset import load_tfrecord_dataset

cfg = load_config("configs/pretrain.yaml")

weights_path = "checkpoints/pretrain/best_weights.weights.h5"
if not os.path.exists(weights_path):
    print("No best-weights file yet. Run step 4 (training) first (best weights are only saved when a validation set exists).")
    raise SystemExit(0)

model = SPICEPretrainModel(cfg.model)
model.load_weights(weights_path)

ds = load_tfrecord_dataset(cfg, "val").take(1)
for x, _ in ds:
    inputs = {"tokens": x["tokens"][None], "env": x["env"][None], "mask": x["mask"][None]}
    out = model(inputs, training=False)
    n = int(tf.reduce_sum(x["mask"]).numpy())
    pred = out["coords"][0, :n].numpy()
    true = x["coords"][:n].numpy()
    L = min(n, 80)
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(true[:L, 0], true[:L, 1], true[:L, 2], "o-", lw=1, label="GT")
    ax.plot(pred[:L, 0], pred[:L, 1], pred[:L, 2], "x--", lw=1, label="Pred")
    ax.legend()
    ax.set_title(f"Cα backbone comparison (first {L} residues)")
    plt.show()
print("Done ✅")

## Next steps

- Full training: `max_shards=0`, `epochs=30`.
- Training curves: `tensorboard --logdir runs/pretrain`.
- Phase 2 (RL): reuse the same trunk, adding Head B/B'/C/D with Rust-engine feedback.